In [ ]:
!pip uninstall community -y
!pip install python-louvain

Found existing installation: community 1.0.0b1
Uninstalling community-1.0.0b1:
  Successfully uninstalled community-1.0.0b1


In [ ]:
import community.community_louvain as community_louvain

In [ ]:
import numpy as np
import networkx as nx
import time
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import SpectralEmbedding
from torch_geometric.datasets import WebKB
from networkx.algorithms.community import louvain_communities

# -------------------------------
# Load WebKB
# -------------------------------
def load_webkb(name="Cornell"):
    dataset = WebKB(root='/tmp/WebKB', name=name)
    data = dataset[0]

    num_nodes = data.num_nodes
    edge_index = data.edge_index.numpy()

    adj = np.zeros((num_nodes, num_nodes))

    for i in range(edge_index.shape[1]):
        u = edge_index[0][i]
        v = edge_index[1][i]
        adj[u][v] = 1

    attributes = data.x.numpy()
    labels = data.y.numpy()

    communities = {}
    for i, label in enumerate(labels):
        communities.setdefault(label, []).append(i)

    return adj, attributes, list(communities.values()), labels


# -------------------------------
# START
# -------------------------------
start_time = time.time()

adj, X, true_communities, y_true = load_webkb("Cornell")

num_nodes = X.shape[0]
num_edges = int(np.sum(adj))

print("Nodes:", num_nodes)
print("Edges:", num_edges)
print("True Communities:", len(true_communities))


# -------------------------------
# Preprocessing
# -------------------------------
X_scaled = StandardScaler().fit_transform(X)
sim_matrix = cosine_similarity(X_scaled)


# -------------------------------
# GENERATIONS
# -------------------------------
best_modularity = -1
best_result = None

generations = 30

for gen in range(generations):

    # 🔥 Smart parameter ranges (not random chaos)
    alpha = np.random.uniform(0.75, 0.9)
    prop = np.random.uniform(0.2, 0.5)
    threshold_pct = np.random.uniform(88, 93)
    resolution = np.random.uniform(0.6, 0.9)

    # -------------------------------
    # Fusion (stable)
    # -------------------------------
    combined = alpha * adj + (1 - alpha) * sim_matrix
    combined += prop * (adj @ sim_matrix)

    # Symmetric
    combined = (combined + combined.T) / 2

    # Clean
    combined[combined < 0] = 0
    combined = np.nan_to_num(combined)

    # Normalize
    max_val = np.max(combined)
    if max_val > 0:
        combined = combined / max_val

    np.fill_diagonal(combined, 1)

    # -------------------------------
    # Spectral Embedding
    # -------------------------------
    embedding = SpectralEmbedding(
        n_components=32,
        affinity='precomputed'
    )
    Z = embedding.fit_transform(combined)

    # -------------------------------
    # Graph from embedding
    # -------------------------------
    sim_embed = cosine_similarity(Z)

    threshold = np.percentile(sim_embed, threshold_pct)
    sim_embed[sim_embed < threshold] = 0

    G = nx.from_numpy_array(sim_embed)

    # -------------------------------
    # Louvain
    # -------------------------------
    communities = louvain_communities(
        G,
        weight='weight',
        resolution=resolution,
        seed=42
    )

    # -------------------------------
    # Merge to 5
    # -------------------------------
    communities = sorted(communities, key=len, reverse=True)
    merged = [set(c) for c in communities[:5]]

    for small_comm in communities[5:]:
        best_idx = 0
        best_score = -1

        for i in range(5):
            score = sum(
                sim_embed[u][v]
                for u in small_comm
                for v in merged[i]
            )

            if score > best_score:
                best_score = score
                best_idx = i

        merged[best_idx].update(small_comm)

    predicted = [list(c) for c in merged]

    # -------------------------------
    # Modularity
    # -------------------------------
    modularity = nx.algorithms.community.quality.modularity(
        G, predicted, weight='weight'
    )

    print(f"Generation {gen+1}: Modularity = {round(modularity,4)}")

    if modularity > best_modularity:
        best_modularity = modularity
        best_result = predicted


# -------------------------------
# END
# -------------------------------
runtime = time.time() - start_time


# -------------------------------
# OUTPUT
# -------------------------------
print("\n===== BEST RESULT AFTER 30 GENERATIONS =====")

print("Nodes:", num_nodes)
print("Edges:", num_edges)
print("Communities (True):", len(true_communities))
print("Communities (Predicted):", len(best_result))

print("\nBest Modularity:", round(best_modularity, 4))
print("Runtime (seconds):", round(runtime, 4))

print("\nCluster Sizes:")
for i, comm in enumerate(best_result):
    print(f"Cluster {i}: {len(comm)} nodes")

Processing...
Done!


Nodes: 183
Edges: 298
True Communities: 5
Generation 1: Modularity = 0.4669
Generation 2: Modularity = 0.4782
Generation 3: Modularity = 0.4199
Generation 4: Modularity = 0.4063
Generation 5: Modularity = 0.436
Generation 6: Modularity = 0.5047
Generation 7: Modularity = 0.4417
Generation 8: Modularity = 0.4799
Generation 9: Modularity = 0.3379
Generation 10: Modularity = 0.4719
Generation 11: Modularity = 0.5343
Generation 12: Modularity = 0.404
Generation 13: Modularity = 0.4135
Generation 14: Modularity = 0.4199
Generation 15: Modularity = 0.4824
Generation 16: Modularity = 0.4632
Generation 17: Modularity = 0.4191
Generation 18: Modularity = 0.5214
Generation 19: Modularity = 0.529
Generation 20: Modularity = 0.4745
Generation 21: Modularity = 0.3966
Generation 22: Modularity = 0.4572
Generation 23: Modularity = 0.5579
Generation 24: Modularity = 0.453
Generation 25: Modularity = 0.5337
Generation 26: Modularity = 0.4483
Generation 27: Modularity = 0.5054
Generation 28: Modularity 

In [ ]:
!pip install torch
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:00
